# 얼굴 검출 기반 딥페이크 탐지 모델 개발

DACON Hecto AI Challenge 참가 프로젝트

---

## 파이프라인 구성

1. **전처리** — MTCNN 기반 얼굴 검출 및 크롭, 이미지 데이터셋 구축
2. **학습** — EfficientNet-B3 transfer learning, augmentation, early stopping

## 핵심 설계 결정

- **MTCNN으로 얼굴만 크롭한 이유**: 원본 비디오를 그대로 사용하면 배경 정보가 많아 학습 효율이 떨어지고 전처리 비용이 큼. 가장 큰 얼굴 하나만 추출해 얼굴 중심 데이터셋으로 재구성
- **30프레임마다 1장 샘플링**: 인접 프레임 간 중복 정보를 줄이고 처리 시간 단축
- **crop margin 30%**: 너무 타이트하게 자르면 턱·이마 등 중요한 facial cue가 잘려 학습에 악영향
- **EfficientNet-B3 선택**: B0는 너무 가볍고 B4 이상은 배치 크기를 8 이하로 줄여야 해 메모리 제약 상 B3가 최적
- **JPEG compression augmentation 추가**: 딥페이크 영상은 압축 과정에서 고유한 아티팩트가 생김. 이를 학습에 반영해 모델이 압축 아티팩트에 과의존하지 않도록 일반화

---
## 1. 전처리 — MTCNN 기반 얼굴 검출 및 크롭

Celeb-DF-v2 비디오 데이터에서 MTCNN으로 얼굴 영역을 검출하고,
30% margin을 추가해 크롭한 뒤 JPG로 저장한다.

**데이터셋 구조**
```
Celeb-DF-v2/
├── Celeb-real/       ← 진짜 (label: 0)
├── YouTube-real/     ← 진짜 (label: 0)
└── Celeb-synthesis/  ← 딥페이크 (label: 1)
```

In [7]:
import cv2
import os
import torch
from facenet_pytorch import MTCNN
from glob import glob
from tqdm import tqdm

INPUT_ROOT  = r'./train_data/Celeb-DF-v2'
OUTPUT_ROOT = r'./train_data/cropped_faces/Celeb-DF-v2'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

# keep_all=False, select_largest=True: 여러 얼굴 중 가장 큰 얼굴 하나만 추출
mtcnn = MTCNN(keep_all=False, select_largest=True, device=device)

def process_video(video_path, save_dir):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return

    frame_count = 0
    save_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 30프레임마다 1장 샘플링 (인접 프레임 중복 방지 + 처리 시간 단축)
        if frame_count % 30 == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            boxes, _ = mtcnn.detect(frame_rgb)

            if boxes is not None:
                box = boxes[0]
                x1, y1, x2, y2 = [int(b) for b in box]
                h, w, _ = frame.shape

                # crop margin 30%: 턱·이마 등 중요한 facial cue가 잘리지 않도록
                mw = int((x2 - x1) * 0.3)
                mh = int((y2 - y1) * 0.3)
                nx1 = max(0, x1 - mw)
                ny1 = max(0, y1 - mh)
                nx2 = min(w, x2 + mw)
                ny2 = min(h, y2 + mh)

                face_img = frame[ny1:ny2, nx1:nx2]
                if face_img.size > 0:
                    save_path = os.path.join(save_dir, f"frame_{frame_count:05d}.jpg")
                    cv2.imwrite(save_path, face_img)
                    save_count += 1

        frame_count += 1

    cap.release()

# Celeb-DF-v2 전체 처리
video_files = glob(os.path.join(INPUT_ROOT, '**/*.mp4'), recursive=True)
print(f"총 {len(video_files)}개의 동영상을 찾았습니다.")

for video_path in tqdm(video_files):
    relative_path = os.path.relpath(os.path.dirname(video_path), INPUT_ROOT)
    save_dir = os.path.join(OUTPUT_ROOT, relative_path)
    os.makedirs(save_dir, exist_ok=True)
    try:
        process_video(video_path, save_dir)
    except Exception as e:
        print(f"Error on {video_path}: {e}")

print("전처리 완료. cropped_faces 폴더를 확인하세요.")

ModuleNotFoundError: No module named 'facenet_pytorch'

---
## 2. 학습 — EfficientNet-B3 Transfer Learning

전처리된 얼굴 이미지로 이진 분류 모델을 학습한다.

| 구성 요소 | 선택 | 이유 |
|-----------|------|------|
| 모델 | EfficientNet-B3 | 메모리 제약 상 B3가 성능/비용 균형 최적 |
| 옵티마이저 | AdamW | weight decay로 과적합 방지 |
| 스케줄러 | CosineAnnealingWarmRestarts | 학습률을 주기적으로 리셋해 local minima 탈출 |
| Early Stopping | patience=5 | 제한된 자원 환경에서 불필요한 학습 방지 |

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from glob import glob
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import random

# ── 하이퍼파라미터 (여기만 수정하면 됨) ──────────────────────────
class CFG:
    SEED          = 42
    IMG_SIZE      = 224
    EPOCHS        = 30       # early stopping 있으므로 넉넉하게 설정
    BATCH_SIZE    = 16       # B3 기준 16이 메모리 한계
    LEARNING_RATE = 1e-4
    EARLY_STOP    = 5

def seed_everything(seed):
    """재현성 확보를 위한 시드 고정"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(CFG.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

### Augmentation 전략

- **학습용**: 다양한 변형으로 모델을 강건하게 학습
  - `ImageCompression`: 딥페이크 영상은 압축 시 고유한 아티팩트가 생김. 이를 augmentation으로 추가해 모델이 압축 아티팩트에 과의존하지 않도록 일반화
- **검증용**: 크기 조정과 정규화만 적용 (공정한 평가를 위해 변형 없음)

In [ ]:
# 학습용 augmentation
train_transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.GaussianBlur(p=0.2),
    # 딥페이크 압축 아티팩트 재현
    A.ImageCompression(quality_lower=60, quality_upper=100, p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(p=1.0),
        A.HueSaturationValue(p=1.0),
    ], p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 검증용: 변형 없이 크기·정규화만
val_transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels     = labels
        self.transform  = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.file_paths[idx])
        if img is None:
            img = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(self.labels[idx], dtype=torch.long)


# 데이터 로드
DATA_ROOT = r'./train_data/cropped_faces/Celeb-DF-v2'

real_paths = (glob(os.path.join(DATA_ROOT, 'Celeb-real', '*.jpg')) +
              glob(os.path.join(DATA_ROOT, 'YouTube-real', '*.jpg')))
fake_paths = glob(os.path.join(DATA_ROOT, 'Celeb-synthesis', '*.jpg'))

print(f"Real: {len(real_paths)} | Fake: {len(fake_paths)}")

all_paths  = real_paths + fake_paths
all_labels = [0] * len(real_paths) + [1] * len(fake_paths)

# Stratify split: real/fake 비율 유지
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.2, random_state=CFG.SEED, stratify=all_labels
)

train_loader = DataLoader(
    FaceDataset(train_paths, train_labels, transform=train_transform),
    batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    FaceDataset(val_paths, val_labels, transform=val_transform),
    batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=0
)

### 모델 구성

ImageNet pretrained EfficientNet-B3의 마지막 분류 레이어만 교체 (real/fake 2-class).
전체 가중치를 fine-tuning해 딥페이크 탐지 태스크에 맞게 학습한다.

In [ ]:
# EfficientNet-B3: 마지막 레이어만 2-class로 교체
model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=1e-5)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

In [ ]:
best_score    = 0.0
patience_count = 0

print(f"학습 시작 (Max Epochs: {CFG.EPOCHS}, Early Stop patience: {CFG.EARLY_STOP})")

for epoch in range(CFG.EPOCHS):
    # ── Train ─────────────────────────────────────────────────────
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG.EPOCHS} [Train]")
    for inputs, labels in train_bar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()
        train_bar.set_postfix(loss=running_loss/total, acc=100.*correct/total)

    scheduler.step()

    # ── Validation ────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            val_loss    += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            val_total   += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f"[Epoch {epoch+1}] Train Acc: {100*correct/total:.2f}% | Val Acc: {val_acc:.2f}% | Val Loss: {val_loss/len(val_loader):.4f}")

    # ── Early Stopping ────────────────────────────────────────────
    if val_acc > best_score:
        best_score     = val_acc
        patience_count = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ✓ Best score updated ({best_score:.2f}%) → 모델 저장")
    else:
        patience_count += 1
        print(f"  patience: {patience_count}/{CFG.EARLY_STOP}")
        if patience_count >= CFG.EARLY_STOP:
            print("Early Stopping 발동")
            break

print(f"학습 완료. Best Val Acc: {best_score:.2f}%")